In [ ]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [ ]:
class SimpleAtomEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(64, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(3,  hidden_dim),
            nn.Embedding(10, hidden_dim),
        ])
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x   = x.long().clamp(min=0)
        out = sum(emb(x[..., i]) for i, emb in enumerate(self.embeddings))
        return self.proj(F.gelu(out))


class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=3):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))


class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered) * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))


class GatedPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, mask):
        scores  = self.gate(x).squeeze(-1)
        scores  = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)


class HybridGraphFNet_Best(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=10,
                 num_heads=4, lap_k=8, dropout=0.1, edge_dim=3):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)
        self.input_proj = SimpleAtomEncoder(hidden_dim)
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'local':  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                'global': SpectralMixMH(hidden_dim, num_heads=num_heads),
                'gate':   nn.Linear(hidden_dim, hidden_dim),
                'norm':   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])
        self.pool = GatedPooling(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim)
        )

    def compute_laplacian_basis(self, adj, mask):
        B, N, _ = adj.shape
        A_list, U_list = [], []
        for b in range(B):
            n        = int(mask[b].sum().item())
            adj_b    = adj[b, :n, :n]
            deg      = adj_b.sum(dim=1)
            D_inv_s  = torch.diag(torch.pow(deg + 1e-8, -0.5))
            A_norm_b = D_inv_s @ adj_b @ D_inv_s
            L_b      = torch.eye(n, device=adj.device) - A_norm_b
            try:
                _, U_b = torch.linalg.eigh(L_b)
                idx    = torch.abs(U_b).argmax(dim=0)
                signs  = torch.sign(U_b[idx, torch.arange(U_b.size(1), device=adj.device)])
                signs[signs == 0] = 1.0
                U_b    = U_b * signs.unsqueeze(0)
            except Exception:
                U_b = torch.eye(n, device=adj.device)
            A_list.append(F.pad(A_norm_b, (0, N-n, 0, N-n)))
            U_list.append(F.pad(U_b,      (0, N-n, 0, N-n)))
        return torch.stack(A_list), torch.stack(U_list)

    def forward(self, data):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj     = to_dense_adj(data.edge_index, data.batch,
                               max_num_nodes=x.size(1))
        if data.edge_attr is not None:
            edge_attr_dense = to_dense_adj(
                data.edge_index, data.batch,
                edge_attr=data.edge_attr[:, :3].float(),
                max_num_nodes=x.size(1)
            )
        else:
            edge_attr_dense = None

        I   = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj = adj + I
        A_norm, U = self.compute_laplacian_basis(adj, mask)

        x = self.input_proj(x)
        k = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        if k < self.lap_k:
            lap_pe = F.pad(lap_pe, (0, self.lap_k - k))
        x = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            x_res    = x
            x_local  = layer['local'](x, A_norm, edge_attr_dense)
            x_global = layer['global'](x, U, mask)
            gate     = torch.sigmoid(layer['gate'](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer['norm'](x_res + self.dropout(x_mix))

        x         = x * mask.unsqueeze(-1)
        graph_emb = self.pool(x, mask)
        return self.classifier(graph_emb)


class HybridGraphFNet_Best_Peptides(HybridGraphFNet_Best):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=10):
        super().__init__(
            hidden_dim=hidden_dim, num_layers=num_layers, out_dim=out_dim,
            num_heads=4, lap_k=8, dropout=0.1, edge_dim=3,
        )

print('Model defined.')

Model defined.


In [ ]:
test_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='test')
test_loader  = DataLoader(test_dataset, batch_size=32)
print(f'Test set: {len(test_dataset)} graphs')

model = HybridGraphFNet_Best_Peptides(
    hidden_dim=128, num_layers=4, out_dim=10
).to(device)

state_dict = torch.load(
    'best_model_func_Best_Peptides_seed1.pt',
    map_location=device
)

# Check what alphas is before ignoring it
if 'alphas' in state_dict:
    print(f'Found alphas in checkpoint: shape={state_dict["alphas"].shape}, values={state_dict["alphas"]}')

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'Missing keys:    {missing}')
print(f'Unexpected keys: {unexpected}')
model.eval()
print(f'Checkpoint loaded.')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

Test set: 2331 graphs
Found alphas in checkpoint: shape=torch.Size([4]), values=tensor([0.5000, 0.5000, 0.5000, 0.5000], device='cuda:0')
Missing keys:    []
Unexpected keys: ['alphas']
Checkpoint loaded.
Params: 328,603


In [ ]:
!pip install torch_geometric -q

from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader

test_dataset = LRGBDataset(root='./data/LRGB', name='Peptides-func', split='test')
test_loader  = DataLoader(test_dataset, batch_size=32)
print(f'Test set: {len(test_dataset)} graphs')

Test set: 2331 graphs


In [ ]:
def gate_analysis_peptides(model, loader, num_batches=10):
    model.eval()
    layer_gates = [[] for _ in range(len(model.layers))]

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= num_batches:
                break
            batch = batch.to(device)

            x, mask = to_dense_batch(batch.x.float(), batch.batch)
            adj     = to_dense_adj(batch.edge_index, batch.batch,
                                   max_num_nodes=x.size(1))
            if batch.edge_attr is not None:
                edge_attr_dense = to_dense_adj(
                    batch.edge_index, batch.batch,
                    edge_attr=batch.edge_attr[:, :3].float(),
                    max_num_nodes=x.size(1)
                )
            else:
                edge_attr_dense = None

            I   = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
            adj = adj + I
            A_norm, U = model.compute_laplacian_basis(adj, mask)

            x = model.input_proj(x)
            k = min(model.lap_k, U.size(-1))
            lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
            if k < model.lap_k:
                lap_pe = F.pad(lap_pe, (0, model.lap_k - k))
            x = x + model.pe_encoder(lap_pe)

            for i, layer in enumerate(model.layers):
                x_res    = x
                x_local  = layer['local'](x, A_norm, edge_attr_dense)
                x_global = layer['global'](x, U, mask)
                gate     = torch.sigmoid(layer['gate'](x))   # [B, N, H]

                valid     = mask.unsqueeze(-1).float()
                gate_mean = (gate * valid).sum(dim=(0,1)) / valid.sum(dim=(0,1)).clamp(min=1)
                layer_gates[i].append(gate_mean.cpu())       # [H]

                x_mix = gate * x_local + (1 - gate) * x_global
                x     = layer['norm'](x_res + model.dropout(x_mix))

    print('\n--- Gate Analysis: Peptides-func test set ---')
    print(f'{"Layer":>7}  {"Mean gate":>12}  {"Std gate":>12}  {"Interpretation":>30}')
    print('-' * 68)
    results = []
    for i, batches in enumerate(layer_gates):
        all_gates = torch.stack(batches, dim=0)   # [num_batches, H]
        mean_g    = all_gates.mean().item()
        std_g     = all_gates.std().item()

        if mean_g > 0.85:
            interp = 'COLLAPSED → GCN'
        elif mean_g < 0.15:
            interp = 'COLLAPSED → spectral'
        elif std_g < 0.05:
            interp = 'uniform (not per-node)'
        else:
            interp = 'genuine per-node routing'

        print(f'{i:>7}  {mean_g:>12.4f}  {std_g:>12.4f}  {interp:>30}')
        results.append((mean_g, std_g))

    print('-' * 68)
    overall_mean = np.mean([r[0] for r in results])
    overall_std  = np.mean([r[1] for r in results])
    print(f'{"Overall":>7}  {overall_mean:>12.4f}  {overall_std:>12.4f}')
    print()
    print('NeighborsMatch comparison:')
    print('  r=2  gates=[0.51,0.56,0.55,0.54]  acc=100%')
    print('  r=3  gates=[0.48,0.50,0.51,0.49]  acc=100%')
    print('  r=4  gates=[0.52,0.52,0.46,0.46]  acc=100%')
    print('  r=5  gates=[0.50,0.51,0.52,0.51]  acc=51%  ← spectral input degraded')
    return results

peptides_gate_results = gate_analysis_peptides(model, test_loader, num_batches=10)


--- Gate Analysis: Peptides-func test set ---
  Layer     Mean gate      Std gate                  Interpretation
--------------------------------------------------------------------
      0        0.3557        0.3371        genuine per-node routing
      1        0.3406        0.1912        genuine per-node routing
      2        0.4167        0.1537        genuine per-node routing
      3        0.5539        0.1952        genuine per-node routing
--------------------------------------------------------------------
Overall        0.4167        0.2193

NeighborsMatch comparison:
  r=2  gates=[0.51,0.56,0.55,0.54]  acc=100%
  r=3  gates=[0.48,0.50,0.51,0.49]  acc=100%
  r=4  gates=[0.52,0.52,0.46,0.46]  acc=100%
  r=5  gates=[0.50,0.51,0.52,0.51]  acc=51%  ← spectral input degraded
